In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
import os
import shutil
import torch
import torch.nn as nn
from torchvision import transforms
from PIL import Image

# =====================================================================
# 1. CONFIGURATION (Update these paths!)
# =====================================================================
# Where are your unsorted custom images right now?
UNSORTED_FOLDER_PATH = '/content/drive/MyDrive/satellite_image/Custom_Data'

# Where do you want the organized folders to be created?
SORTED_OUTPUT_PATH = '/content/drive/MyDrive/satellite_image/Custom_Data_Results'

# Path to your saved model
BEST_MODEL_PATH = '/content/drive/MyDrive/satellite_image/EuroSAT_Models/dinov2_eurosat_best.pth'

# The 10 EuroSAT categories your model was trained to recognize
CLASSES = [
    'AnnualCrop', 'Forest', 'HerbaceousVegetation', 'Highway',
    'Industrial', 'Pasture', 'PermanentCrop', 'Residential',
    'River', 'SeaLake'
]

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")

# =====================================================================
# 2. LOAD MODEL & TRANSFORMS
# =====================================================================
class DINOv2LinearProbe(nn.Module):
    def __init__(self, model_name='dinov2_vitb14', num_classes=10):
        super().__init__()
        self.backbone = torch.hub.load('facebookresearch/dinov2', model_name)
        for param in self.backbone.parameters():
            param.requires_grad = False

        embed_dim = self.backbone.embed_dim
        self.classifier = nn.Sequential(
            nn.BatchNorm1d(embed_dim),
            nn.Dropout(0.2),
            nn.Linear(embed_dim, num_classes)
        )

    def forward(self, x):
        with torch.no_grad():
            features = self.backbone(x)
        return self.classifier(features)

print("Loading model weights...")
model = DINOv2LinearProbe().to(DEVICE)
model.load_state_dict(torch.load(BEST_MODEL_PATH, map_location=DEVICE))
model.eval()

# Must perfectly match the transforms used during validation/training
inference_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# =====================================================================
# 3. AUTOMATED INFERENCE & SORTING LOGIC
# =====================================================================
def sort_custom_images(source_dir, output_dir):
    if not os.path.exists(source_dir):
        print(f"❌ ERROR: Source directory '{source_dir}' not found.")
        return

    # Create the 10 empty class folders in the output directory
    for cls in CLASSES:
        os.makedirs(os.path.join(output_dir, cls), exist_ok=True)

    valid_extensions = ('.jpg', '.jpeg', '.png', '.bmp', '.tif', '.tiff')
    image_files = [f for f in os.listdir(source_dir) if f.lower().endswith(valid_extensions)]

    if len(image_files) == 0:
        print(f"⚠️ No valid images found in {source_dir}.")
        return

    print(f"\nSorting {len(image_files)} test images into '{output_dir}'...")

    success_count = 0
    with torch.no_grad():
        for filename in image_files:
            img_path = os.path.join(source_dir, filename)
            try:
                # 1. Load and format image
                img = Image.open(img_path).convert("RGB")
                tensor = inference_transforms(img).unsqueeze(0).to(DEVICE)

                # 2. Predict class using mixed precision
                with torch.amp.autocast('cuda'):
                    output = model(tensor)
                    pred_idx = torch.argmax(output, dim=1).item()
                    pred_class = CLASSES[pred_idx]

                # 3. Copy file to the corresponding folder
                dest_path = os.path.join(output_dir, pred_class, filename)
                shutil.copy(img_path, dest_path)
                success_count += 1

            except Exception as err:
                print(f"Failed to process {filename}: {err}")

    print(f"\n✅ Completed! Successfully categorized {success_count} images.")
    print(f"Check your folders at: {output_dir}")

# Run the sorter
sort_custom_images(UNSORTED_FOLDER_PATH, SORTED_OUTPUT_PATH)

Using device: cpu
Loading model weights...


Using cache found in /root/.cache/torch/hub/facebookresearch_dinov2_main



Sorting 50 test images into '/content/drive/MyDrive/satellite_image/Custom_Data_Results'...


/tmp/ipykernel_2385/1872598054.py:95: UserWarning: CUDA is not available or torch_xla is imported. Disabling autocast.
  with torch.amp.autocast('cuda'):



✅ Completed! Successfully categorized 50 images.
Check your folders at: /content/drive/MyDrive/satellite_image/Custom_Data_Results
